In [2]:
from qrisp import *
from qrisp.interface import IQMBackend
from qrisp.grover import tag_state, grovers_alg
from iqm.applications.maxcut import MaxCutInstance
from iqm.qaoa.qubo_qaoa import QUBOQAOA
from iqm.applications.maxcut import maxcut_generator
from iqm.qaoa.circuits import qiskit_circuit

import os
import networkx as nx

In [14]:
if not os.environ.get("IQM_TOKEN"):
    os.environ["IQM_TOKEN"] = input("IQM Resonance token: ")

os.environ["IQM_SERVER_URL"] = "https://resonance.iqm.tech"

backend = IQMBackend(device_instance="garnet")

In [15]:
def initialize_and_Hadamard(length_of_secret_code):
    inputs = QuantumVariable(length_of_secret_code)
    h(inputs)
    return inputs
print(initialize_and_Hadamard(4))

{'0000': 0.0625, '1000': 0.0625, '0100': 0.0625, '1100': 0.0625, '0010': 0.0625, '1010': 0.0625, '0110': 0.0625, '1110': 0.0625, '0001': 0.0625, '1001': 0.0625, '0101': 0.0625, '1101': 0.0625, '0011': 0.0625, '1011': 0.0625, '0111': 0.0625, '1111': 0.0625}


In [16]:
def oracle(qubits, secret_code):
    anc = QuantumVariable(1)
    x(anc)
    h(anc)

    for i in range(len(secret_code)):
        if secret_code[i] == '1':
            cx(qubits[i], anc)


In [17]:
def final_Hadamards(qubits):
    h(qubits)

In [22]:
def BernsteinVazirani(secret_code):
    length_of_secret_code = len(secret_code)
    qubits = initialize_and_Hadamard(length_of_secret_code)
    oracle(qubits, secret_code)
    final_Hadamards(qubits)
    measure(qubits)
    return qubits

secret_code='1001'
inputs = BernsteinVazirani(secret_code)

# print(inputs)
# print(inputs.qs)

circuit = inputs.qs.compile()
job = backend.run_async(circuit)
print(f"Job ID: {job.job_id}")
print(f"Results: {job.result().get_counts()}")

{'1001': 1.0}                                                                        
Job ID: 01a16f7b-cb4b-756c-82eb-36c65fb812a7
Results: {'0000': 19, '1000': 44, '1010': 1, '0001': 74, '1001': 845, '0011': 1, '1011': 16}
